# SQL Business Deep-Dive Analysis Using DuckDB

### Business Context & Objectives
Following the visual insights discovered during Phase 2 (EDA), this notebook formally shifts our analytical execution into an advanced SQL querying environment powered by **DuckDB**. 

Our primary objective is to execute high-precision data metrics to isolate operational leakages, identify financial deficits across categories, and provide actionable numbers for the executive team (CMO, Sales Ops).

### Analytical Infrastructure: Why DuckDB?
* **Serverless Processing:** DuckDB runs locally directly inside our Python kernel, requiring no complex database server configurations or maintenance.
* **Vectorized Columnar Engine:** Engineered specifically for Data Analytics. Aggregation functions like `SUM()` and `AVG()` execute over large sequences within fractions of a millisecond.
* **Seamless In-Memory Integration:** DuckDB can query native Pandas DataFrames as if they were standard relational SQL tables.

In [1]:
import pandas as pd
import duckdb

# Load the high-quality cleaned dataset produced in Phase 1
df_clean = pd.read_csv('../data/cleaned/superstore_orders_cleaned.csv')

# Instantiate a local serverless in-memory DuckDB connection
con = duckdb.connect()

# Register the Pandas DataFrame as a temporary relational table view inside DuckDB
# This dynamically maps the 'df_clean' variable to the SQL table named 'superstore'
con.register('superstore', df_clean)

# Operational check: Run a tiny query to verify the relational infrastructure is working
con.execute("SELECT * FROM superstore LIMIT 1").df()

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408.0,2,0.0,106.14,35.46,Medium,2011


## 1. Product Sub-Category Profitability Leakage

### Business Hypothesis
During Phase 2 (Visual EDA), we spotted that some sub-categories with massive sales volumes might actually yield a negative net profit. This query aggregates total sales and total profits, then calculates the exact **Profit Margin (%)** to identify the top 5 financial drains on the business.

In [2]:
# SQL Query to find the top 5 most unprofitable product sub-categories
query_q1 = """
SELECT 
    sub_category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100, 2) AS profit_margin_pct
FROM superstore
GROUP BY sub_category
ORDER BY total_profit ASC
LIMIT 5;
"""

# Execute the query and load the results into a DataFrame
df_q1 = con.execute(query_q1).df()
df_q1

,sub_category,total_sales,total_profit,profit_margin_pct
0,Tables,289686.0,-64083.39,-22.12
1,Fasteners,83254.0,11525.42,13.84
2,Labels,73433.0,15010.51,20.44
3,Supplies,210069.0,22583.26,10.75
4,Envelopes,170926.0,29601.12,17.32


### 💡 Operational Interpretation
* **The Standalone Financial Culprit:** **"Tables"** is definitively confirmed as the most severe financial leakage point in the company. It is the **only** sub-category in the bottom 5 that generates a net loss, bleeding **-$64,083.39** with a disastrous profit margin of **-22.12%**.
* **Contrast with Other Low-Sales Sub-categories:** Sub-categories like *Fasteners* and *Labels* generate much lower revenue ($73K - $83K) but still maintain healthy positive profit margins (13.84% and 20.44% respectively). This underscores that the issue with "Tables" is not a lack of volume, but an extreme structural pricing or high product cost defect.

## 2. Deep-Dive into Southeast Asia (SEA) Operational Deficit

### Business Hypothesis
Geographic analysis showed that the Southeast Asia region operates at an incredibly low profit margin despite a healthy revenue stream. This targeted query filters specifically for 'Southeast Asia' and breaks down performance by product category to trace the root cause of this localized crisis.

In [7]:
# SQL Query to analyze major category performance inside Southeast Asia
query_q3 = """
SELECT 
    category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100, 2) AS profit_margin_pct
FROM superstore
WHERE region = 'Southeast Asia'
GROUP BY category
ORDER BY total_profit ASC;
"""

# Execute and view results
df_q3 = con.execute(query_q3).df()
df_q3

,category,total_sales,total_profit,profit_margin_pct
0,Furniture,185329.0,-7269.77,-3.92
1,Office Supplies,159330.0,4173.25,2.62
2,Technology,187513.0,20948.84,11.17


### 💡 Operational Interpretation
* **Furniture Driving the Regional Deficit:** Within the Southeast Asia (SEA) region, the **Furniture** category is the root cause of the overall regional margin collapse, pulling in **-$7,269.77** in net loss with a margin of **-3.92%**.
* **Technology is the Savior:** On a positive note, the **Technology** category performs strongly in SEA, capturing over **$20,948.84** in net profit with a solid **11.17%** profit margin. 
* **Strategic Takeaway:** The company should pivot its strategy in SEA by scaling back promotions on Furniture items and shifting the marketing budget toward high-margin Technology hardware to stabilize regional performance.

## 3. Customer Segment Efficiency for Marketing Prioritization

### Business Context
The Marketing Operations team wants to maximize return on ad spend (ROI). Instead of just looking at aggregate total profit, this query calculates an efficiency metric: **Profit per Distinct Order ID** across different customer segments, allowing the CMO to prioritize high-yield customer profiles.

In [5]:
# SQL Query to calculate total orders, total profit, and average profit per single order ID
query_q4 = """
SELECT 
    segment,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND(SUM(profit) / COUNT(DISTINCT order_id), 2) AS profit_per_order
FROM superstore
GROUP BY segment
ORDER BY profit_per_order DESC;
"""

# Execute and view results
df_q4 = con.execute(query_q4).df()
df_q4

,segment,total_orders,total_profit,profit_per_order
0,Home Office,4687,277009.18,59.10
1,Corporate,7673,442785.86,57.71
2,Consumer,13104,749239.78,57.18


### 💡 Operational Interpretation
* **The Efficiency Winner ("Home Office"):** The **Home Office** segment yields the highest business efficiency, generating **$59.10 in net profit per distinct order**. 
* **The Volume Winner ("Consumer"):** While the *Consumer* segment generates the highest aggregate total profit ($749,239.78), its operational efficiency per transaction ($57.18) is slightly lower than Home Office.
* **Strategic Recommendation for CMO:** Since Home Office orders yield higher profit density, the marketing team should prioritize and target this professional segment during high-converting campaigns to capture more profitable orders with lower relative processing costs.

## 4. Root Cause Analysis on "Tables" vs "Copiers" Cost Structure

### Business Objective
To find out WHY "Tables" are bleeding cash. We will evaluate the Average Discount rate given to customers and the Shipping Cost impact relative to Sales, benchmarking against "Copiers" (our most profitable sub-category).

In [8]:
query_q4_root_cause = """
SELECT 
    sub_category,
    ROUND(AVG(discount) * 100, 2) AS avg_discount_pct,
    ROUND(SUM(shipping_cost), 2) AS total_shipping_cost,
    ROUND((SUM(shipping_cost) / SUM(sales)) * 100, 2) AS shipping_to_sales_pct,
    ROUND(AVG(profit), 2) AS avg_profit_per_item
FROM superstore
WHERE sub_category IN ('Tables', 'Copiers')
GROUP BY sub_category;
"""

df_q4_root = con.execute(query_q4_root_cause).df()
df_q4_root

,sub_category,avg_discount_pct,total_shipping_cost,shipping_to_sales_pct,avg_profit_per_item
0,Tables,29.07,79861.46,27.57,-74.43
1,Copiers,11.71,159496.49,20.38,116.31


### 💡 Root Cause Analysis: Why "Tables" Destroy Business Value
By benchmarking the cost structure of **Tables** (our worst sub-category) against **Copiers** (our best sub-category), we discovered a dual operational failure:

1. **Aggressive Over-Discounting (The 20% Threshold Breach):** * **Tables** suffer from an incredibly high **Average Discount of 29.07%**. 
   * Recalling our Phase 2 EDA, we established that a discount rate above 20% drops profitability below the break-even line. The sales team is consistently breaching this safe threshold, cutting away almost 30% of gross value per transaction.
   * In contrast, **Copiers** maintain price integrity with a strictly managed average discount of only **11.71%**.

2. **The Shipping Cost Burden (Logistical Bleeding):**
   * Shipping costs for **Tables** account for a staggering **27.57% of total sales revenue** (Shipping-to-Sales Ratio). Because tables are bulky and heavy furniture items, they incur premium freight charges.
   * Combined together: **29.07% (Discount) + 27.57% (Shipping Cost) = 56.64%**. This means more than half of the item's gross revenue is completely erased before accounting for manufacturing costs (COGS)!
   * This leaves each single table transaction with an **Average Net Loss of -$74.43**, while a single copier transaction brings in a healthy **+$116.31 in profit**.

### 🛠️ Strategic Recommendation for Executive Team
* **Immediate Action:** The current discount structure for Tables is completely unsustainable. The sales department must immediately disable autonomous discounting for the Tables sub-category in CRM systems, enforcing a strict maximum discount limit of 10%.
* **Logistics Realignment:** Renegotiate bulk shipping contracts for large-freight furniture or pivot towards a "Customer Collects" model to decrease the 27.57% shipping burden.

## 5. Southeast Asia Performance Breakdown by Country

### Business Objective
We know Southeast Asia has a margin issue. This query isolates individual countries within the SEA region to pinpoint exactly where the operational leakage is happening.

In [10]:
# SQL Query 5: Corrected Syntax for SEA Country Breakdown
query_q5_sea_countries = """
SELECT 
    country,
    COUNT(order_id) AS total_transactions,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100, 2) AS profit_margin_pct
FROM superstore
WHERE region = 'Southeast Asia'
GROUP BY country
ORDER BY total_profit ASC;
"""

# Execute the query through DuckDB and fetch as Pandas DataFrame
df_q5_sea = con.execute(query_q5_sea_countries).df()
df_q5_sea

,country,total_transactions,total_sales,total_profit,profit_margin_pct
0,Philippines,681,120135.0,-16128.22,-13.43
1,Thailand,295,41102.0,-7308.20,-17.78
2,Myanmar (Burma),136,26341.0,-2109.26,-8.01
3,Vietnam,265,47572.0,-1870.23,-3.93
4,Cambodia,45,8371.0,4476.54,53.48
5,Singapore,141,28693.0,8853.06,30.85
6,Indonesia,1390,224180.0,15608.68,6.96
7,Malaysia,176,35778.0,16329.96,45.64


### 💡 Advanced Strategic Interpretation (The SEA Margin Dichotomy)
A granular breakdown of the Southeast Asia (SEA) region reveals that the geographical deficit is not uniform. Instead, the region is highly fragmented into two distinct clusters: **The Profit Drains** versus **The High-Efficiency Drivers**.

#### 1. The High-Volume Profit Drains (Philippines & Thailand)
* **The Philippines Crisis:** The **Philippines** represents the most critical operational failure in the region. Despite generating **$120,135.00** in revenue across 681 transactions, it produced a massive net loss of **-$16,128.22** with a highly alarming profit margin of **-13.43%**. 
* **Thailand's Margin Collapse:** **Thailand** presents the worst profitability efficiency. It bleeds **-$7,308.20** on a smaller sales baseline ($41,102.00), translating to the region's lowest profit margin at **-17.78%**. 
* **Business Hypothesis:** Sales teams in the Philippines and Thailand are likely engaged in destructive price wars, offering unsustainable promotion packages, or encountering localized supply chain disruptions that artificially inflate cross-border freight costs.

#### 2. The High-Efficiency Drivers (Malaysia, Cambodia, & Singapore)
* **Malaysia's Outstanding Performance:** In sharp contrast, **Malaysia** is a star performer, bringing in **+$16,329.96** in net profit on only $35,778.00 in sales, achieving a staggering profit margin of **45.64%**.
* **Cambodia's High Density:** **Cambodia** operates at maximum efficiency, pulling in a phenomenal **53.48% profit margin**, meaning more than half of every dollar generated directly turns into net cash.
* **Singapore's Stability:** **Singapore** remains highly lucrative, securing a stable **30.85% profit margin**.

### 🛠️ Executive Recommendations for Regional Directors
* **Halt Promotional Spending in the Philippines/Thailand:** Apply an immediate freeze on general discount campaigns within the Philippines and Thailand. Pivot the pricing algorithm toward a cost-plus model until a full tariff and local cost audit is conducted.
* **Capitalize on Malaysia and Cambodia:** Reallocate the regional marketing budget away from underperforming territories and heavily invest in expanding market share in Malaysia and Cambodia, where the business structure safely converts volume into bottom-line profits.

# Conclusion: Executive Summary Matrix

| Analytical Focus | The Culprit / Opportunity | Data Verdict (Key Metrics) | Core Root Cause | Strategic Action |
| :--- | :--- | :--- | :--- | :--- |
| **Product Sub-Category** | **Tables** (Culprit) | Sales: \$289.6K <br>Profit: **-\$64.1K** <br>Margin: **-22.12%** | Unmanaged Average Discount of **29.07%** combined with a punishing **27.57%** Shipping-to-Sales ratio. | Enforce a strict 10% hard discount limit in CRM and redesign bulk shipping logistics. |
| **Regional Category** | **Furniture in SEA** (Culprit) | Sales: \$185.3K <br>Profit: **-\$7.2K** <br>Margin: **-3.92%** | Disproportionate operational push on low-margin Furniture instead of lucrative Technology items. | De-escalate Furniture promotions in SEA; shift capital to scale the **Technology** segment (11.17% margin). |
| **Customer Segment** | **Home Office** (Opportunity) | Profit/Order: **\$59.10** <br>Total Orders: 4,687 | Highest business efficiency per transaction; lower price sensitivity compared to Consumer segments. | Prioritize Home Office professional profiles during digital marketing ad spend to maximize ROI. |
| **Geographic Country** | **Philippines & Thailand** (Culprit) | Ph Margin: **-13.43%** <br>TH Margin: **-17.78%** | Local market discount saturation or heavy distribution inefficiencies eating up gross margins. | Enact immediate margin protections in PH/TH; reallocate capital into highly profitable **Malaysia** (45.64% margin). |

---
*End of Phase 3 Analysis. The relational data warehouse insights established here will serve as the programmatic framework for our downstream Executive Dashboard design.*